In [ ]:
!pip install anthropic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 390.3/390.3 kB 12.1 MB/s eta 0:00:00


In [ ]:
import os
import time
import json
import re
import pandas as pd
from datasets import load_dataset
from tqdm import tqdm
from anthropic import Anthropic
from google.colab import userdata

In [ ]:
from google.colab import drive

# mount drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
MODEL_NAME = "claude-haiku-4-5-20251001"
BATCH_SIZE = 5
SLEEP_TIME = 1.2

OUTPUT_BASE = "/content/drive/MyDrive/fyp-2025/Datasets/PairwiseEvalProcessed/"
os.makedirs(OUTPUT_BASE, exist_ok=True)

claude_api = userdata.get("claude_api")
anthropic = Anthropic(api_key=claude_api)

#### Pairwise Evaluation Prompt

In [ ]:
PAIRWISE_EVAL_PROMPT = """
You are an expert evaluator specializing in customer-service interactions.

Your task is to compare two generated responses (Response A and Response B) to the same client query
and conversation context. Both responses were produced by different AI systems acting as
professional customer-service agents.

Use the Client Question and the Conversation History Summary as the main context for evaluation.
Use the full Conversation History only if additional background is needed.

A Reference Agent Response is provided only as an example of a good customer-service reply.
It is for general guidance only and should NOT be used as a comparison target.
Do not judge responses based on how similar they are to the reference.

Evaluation Criteria (all criteria are equally important):

1. Human-Likeness:
Which response sounds more natural and human-like in a real customer-service conversation?

2. Continuity and Context Understanding:
Which response better reflects the earlier conversation and correctly uses relevant details?

3. Tone and Clarity:
Which response is more professional, polite, empathetic, and easy to understand?

4. Task Appropriateness:
Which response better addresses the client’s request while following realistic customer-service practices?

If both responses are very similar in quality, choose the one that feels slightly more natural and
better connected to the context.

Context Inputs:
Conversation History: {history}
Conversation Summarized History: {history_summary}
Client Question: {client_question}
Reference Agent Response (example only): {ground_truth}

Response A:
{response_a}

Response B:
{response_b}

Choose the single better response overall.

Return your answer as valid JSON only.
Do not include any explanation or extra text.

Output Format:
{{ "winner": "A" or "B" }}
"""

In [ ]:
# Helper Function
def clean_json(text):
    text = text.strip()
    text = re.sub(r"^```[a-zA-Z]*\s*", "", text)
    text = re.sub(r"\s*```$", "", text)
    return text.replace("\ufeff", "").replace("\u200b", "").strip()

def is_unjudged(x):
    return pd.isna(x) or str(x).strip() == "" or str(x) == "None"

def extract_model_name(col):
    return col.replace("generated_answer_", "")

In [ ]:
def run_pairwise_eval(dataset_repo: str):
    print(f"\n=== Running PAIREVAL on {dataset_repo} ===")

    output_csv = os.path.join(
        OUTPUT_BASE,
        dataset_repo.split("/")[-1] + "_processed.csv"
    )

    # Load or resume
    if os.path.exists(output_csv):
        df = pd.read_csv(output_csv)
        print("Resuming from existing processed file")
    else:
        df = load_dataset(dataset_repo, split="train").to_pandas()
        for c in ["judge_pass_1", "judge_pass_2", "final_result"]:
            if c not in df.columns:
                df[c] = ""
        df.to_csv(output_csv, index=False)

    # Detect model outputs
    answer_cols = [c for c in df.columns if c.startswith("generated_answer_")]
    if len(answer_cols) != 2:
        raise ValueError("Dataset must contain exactly two generated_answer_* columns")

    col_a, col_b = answer_cols
    model_a = extract_model_name(col_a)
    model_b = extract_model_name(col_b)

    print(f"Model A: {model_a}")
    print(f"Model B: {model_b}")

    pending_idx = df.index[
        df["judge_pass_1"].apply(is_unjudged) |
        df["judge_pass_2"].apply(is_unjudged)
    ]

    print(f"Pending rows: {len(pending_idx)}")

    batch = 0

    for idx in tqdm(pending_idx, desc="Evaluating rows", unit="row"):
        row = df.loc[idx]

        try:
            # PASS 1 (A vs B)
            if is_unjudged(row["judge_pass_1"]):
                prompt = PAIRWISE_EVAL_PROMPT.format(
                    history=row["history"],
                    history_summary=row["history_summary"],
                    client_question=row["client_question"],
                    ground_truth=row["ground_truth"],
                    response_a=row[col_a],
                    response_b=row[col_b],
                )

                r = anthropic.messages.create(
                    model=MODEL_NAME,
                    temperature=0,
                    max_tokens=100,
                    messages=[{"role": "user", "content": prompt}],
                )

                df.at[idx, "judge_pass_1"] = json.loads(
                    clean_json(r.content[0].text)
                )["winner"]

            # PASS 2 (B vs A)
            if is_unjudged(row["judge_pass_2"]):
                prompt = PAIRWISE_EVAL_PROMPT.format(
                    history=row["history"],
                    history_summary=row["history_summary"],
                    client_question=row["client_question"],
                    ground_truth=row["ground_truth"],
                    response_a=row[col_b],
                    response_b=row[col_a],
                )

                r = anthropic.messages.create(
                    model=MODEL_NAME,
                    temperature=0,
                    max_tokens=50,
                    messages=[{"role": "user", "content": prompt}],
                )

                w = json.loads(clean_json(r.content[0].text))["winner"]
                df.at[idx, "judge_pass_2"] = "A" if w == "B" else "B"

            # FINAL RESULT
            p1 = df.at[idx, "judge_pass_1"]
            p2 = df.at[idx, "judge_pass_2"]

            if p1 == p2 == "A":
                df.at[idx, "final_result"] = model_a
            elif p1 == p2 == "B":
                df.at[idx, "final_result"] = model_b
            else:
                df.at[idx, "final_result"] = "Tie"

            batch += 1
            if batch >= BATCH_SIZE:
                df.to_csv(output_csv, index=False)
                batch = 0

            time.sleep(SLEEP_TIME)

        except Exception as e:
            print(f"SKIPPING row {idx} due to error: {e}")
            df.at[idx, "final_result"] = "Error"
            df.to_csv(output_csv, index=False)
            continue

    df.to_csv(output_csv, index=False)
    print(f"Finished {dataset_repo}")

### gpt4.1-vs-qwen3-4b

In [ ]:
run_pairwise_eval("Lakshan2003/pairwise-gpt4.1-vs-qwen3-4b")


=== Running PAIREVAL on Lakshan2003/pairwise-gpt4.1-vs-qwen3-4b ===
Resuming from existing processed file
Model A: gpt_4_1
Model B: qwen3_4b
Pending rows: 0


Evaluating rows: 0row [00:00, ?row/s]

Finished Lakshan2003/pairwise-gpt4.1-vs-qwen3-4b


### pairwise-gpt4.1-vs-phi-4-mini

In [ ]:
run_pairwise_eval("Lakshan2003/pairwise-gpt4.1-vs-phi-4-mini")


=== Running PAIREVAL on Lakshan2003/pairwise-gpt4.1-vs-phi-4-mini ===
Resuming from existing processed file
Model A: gpt_4_1
Model B: phi_4_mini
Pending rows: 0


Evaluating rows: 0row [00:00, ?row/s]

Finished Lakshan2003/pairwise-gpt4.1-vs-phi-4-mini


### gpt4.1-vs-llama-3.2-instruct

In [ ]:
run_pairwise_eval("Lakshan2003/pairwise-gpt4.1-vs-llama-3.2-instruct")


=== Running PAIREVAL on Lakshan2003/pairwise-gpt4.1-vs-llama-3.2-instruct ===
Resuming from existing processed file
Model A: gpt_4_1
Model B: llama3_2
Pending rows: 0


Evaluating rows: 0row [00:00, ?row/s]


Finished Lakshan2003/pairwise-gpt4.1-vs-llama-3.2-instruct


### gemini-2.5-flash-vs-qwen3-4b

In [ ]:
run_pairwise_eval("Lakshan2003/pairwise-gemini-2.5-flash-vs-qwen3-4b")


=== Running PAIREVAL on Lakshan2003/pairwise-gemini-2.5-flash-vs-qwen3-4b ===
Resuming from existing processed file
Model A: gemini_2_5_flash
Model B: qwen3_4b
Pending rows: 0


Evaluating rows: 0row [00:00, ?row/s]


Finished Lakshan2003/pairwise-gemini-2.5-flash-vs-qwen3-4b


### gemini-2.5-flash-vs-phi-4-mini

In [ ]:
run_pairwise_eval("Lakshan2003/pairwise-gemini-2.5-flash-vs-phi-4-mini")


=== Running PAIREVAL on Lakshan2003/pairwise-gemini-2.5-flash-vs-phi-4-mini ===
Resuming from existing processed file
Model A: gemini_2_5_flash
Model B: phi_4_mini
Pending rows: 0


Evaluating rows: 0row [00:00, ?row/s]

Finished Lakshan2003/pairwise-gemini-2.5-flash-vs-phi-4-mini


### gemini-2.5-flash-vs-llama-3.2-instruct

In [ ]:
run_pairwise_eval("Lakshan2003/pairwise-gemini-2.5-flash-vs-llama-3.2-instruct")


=== Running PAIREVAL on Lakshan2003/pairwise-gemini-2.5-flash-vs-llama-3.2-instruct ===
Resuming from existing processed file
Model A: gemini_2_5_flash
Model B: llama3_2
Pending rows: 0


Evaluating rows: 0row [00:00, ?row/s]

Finished Lakshan2003/pairwise-gemini-2.5-flash-vs-llama-3.2-instruct


### virtuoso-large-vs-qwen3-4b

In [ ]:
run_pairwise_eval("Lakshan2003/pairwise-virtuoso-large-vs-qwen3-4b")


=== Running PAIREVAL on Lakshan2003/pairwise-virtuoso-large-vs-qwen3-4b ===
Resuming from existing processed file
Model A: virtuoso_large
Model B: qwen3_4b
Pending rows: 0


Evaluating rows: 0row [00:00, ?row/s]

Finished Lakshan2003/pairwise-virtuoso-large-vs-qwen3-4b


### virtuoso-large-vs-phi-4-mini

In [ ]:
run_pairwise_eval("Lakshan2003/pairwise-virtuoso-large-vs-phi-4-mini")


=== Running PAIREVAL on Lakshan2003/pairwise-virtuoso-large-vs-phi-4-mini ===
Resuming from existing processed file
Model A: virtuoso_large
Model B: phi_4_mini
Pending rows: 0


Evaluating rows: 0row [00:00, ?row/s]

Finished Lakshan2003/pairwise-virtuoso-large-vs-phi-4-mini


### virtuoso-large-vs-llama-3.2-instruct

In [ ]:
run_pairwise_eval("Lakshan2003/pairwise-virtuoso-large-vs-llama-3.2-instruct")


=== Running PAIREVAL on Lakshan2003/pairwise-virtuoso-large-vs-llama-3.2-instruct ===
Resuming from existing processed file
Model A: virtuoso_large
Model B: llama3_2
Pending rows: 0


Evaluating rows: 0row [00:00, ?row/s]


Finished Lakshan2003/pairwise-virtuoso-large-vs-llama-3.2-instruct


### gpt4.1-vs-qwen3-8b

In [ ]:
run_pairwise_eval("Lakshan2003/pairwise-gpt4.1-vs-qwen3-8b")


=== Running PAIREVAL on Lakshan2003/pairwise-gpt4.1-vs-qwen3-8b ===
Resuming from existing processed file
Model A: gpt_4_1
Model B: qwen3_8b
Pending rows: 0


Evaluating rows: 0row [00:00, ?row/s]

Finished Lakshan2003/pairwise-gpt4.1-vs-qwen3-8b


### gpt4.1-vs-llama3.1-8b

In [ ]:
run_pairwise_eval("Lakshan2003/pairwise-gpt4.1-vs-llama3.1-8b")


=== Running PAIREVAL on Lakshan2003/pairwise-gpt4.1-vs-llama3.1-8b ===
Resuming from existing processed file
Model A: gpt_4_1
Model B: llama3_1_8b
Pending rows: 0


Evaluating rows: 0row [00:00, ?row/s]

Finished Lakshan2003/pairwise-gpt4.1-vs-llama3.1-8b


### gpt4.1-vs-qwen3-1.7b

In [ ]:
run_pairwise_eval("Lakshan2003/pairwise-gpt4.1-vs-qwen3-1.7b")


=== Running PAIREVAL on Lakshan2003/pairwise-gpt4.1-vs-qwen3-1.7b ===
Resuming from existing processed file
Model A: gpt_4_1
Model B: qwen3_1_7b
Pending rows: 0


Evaluating rows: 0row [00:00, ?row/s]


Finished Lakshan2003/pairwise-gpt4.1-vs-qwen3-1.7b


### gpt4.1-vs-llama3.2-1b

In [ ]:
run_pairwise_eval("Lakshan2003/pairwise-gpt4.1-vs-llama3.2-1b")


=== Running PAIREVAL on Lakshan2003/pairwise-gpt4.1-vs-llama3.2-1b ===
Resuming from existing processed file
Model A: gpt_4_1
Model B: llama3_2_1b
Pending rows: 0


Evaluating rows: 0row [00:00, ?row/s]

Finished Lakshan2003/pairwise-gpt4.1-vs-llama3.2-1b


### gemini-2.5-flash-vs-qwen3-8b

In [ ]:
run_pairwise_eval("Lakshan2003/pairwise-gemini-2.5-flash-vs-qwen3-8b")


=== Running PAIREVAL on Lakshan2003/pairwise-gemini-2.5-flash-vs-qwen3-8b ===
Resuming from existing processed file
Model A: gemini_2_5_flash
Model B: qwen3_8b
Pending rows: 0


Evaluating rows: 0row [00:00, ?row/s]

Finished Lakshan2003/pairwise-gemini-2.5-flash-vs-qwen3-8b


### gemini-2.5-flash-vs-llama3.1-8b

In [ ]:
run_pairwise_eval("Lakshan2003/pairwise-gemini-2.5-flash-vs-llama3.1-8b")


=== Running PAIREVAL on Lakshan2003/pairwise-gemini-2.5-flash-vs-llama3.1-8b ===
Resuming from existing processed file
Model A: gemini_2_5_flash
Model B: llama3_1_8b
Pending rows: 0


Evaluating rows: 0row [00:00, ?row/s]

Finished Lakshan2003/pairwise-gemini-2.5-flash-vs-llama3.1-8b


### gemini-2.5-flash-vs-qwen3-1.7

In [ ]:
run_pairwise_eval("Lakshan2003/pairwise-gemini-2.5-flash-vs-qwen3-1.7b")


=== Running PAIREVAL on Lakshan2003/pairwise-gemini-2.5-flash-vs-qwen3-1.7b ===
Resuming from existing processed file
Model A: gemini_2_5_flash
Model B: qwen3_1_7b
Pending rows: 0


Evaluating rows: 0row [00:00, ?row/s]

Finished Lakshan2003/pairwise-gemini-2.5-flash-vs-qwen3-1.7b


### gemini-2.5-flash-vs-llama3.2-1b

In [ ]:
run_pairwise_eval("Lakshan2003/pairwise-gemini-2.5-flash-vs-llama3.2-1b")


=== Running PAIREVAL on Lakshan2003/pairwise-gemini-2.5-flash-vs-llama3.2-1b ===
Resuming from existing processed file
Model A: gemini_2_5_flash
Model B: llama3_2_1b
Pending rows: 0


Evaluating rows: 0row [00:00, ?row/s]

Finished Lakshan2003/pairwise-gemini-2.5-flash-vs-llama3.2-1b


#### virtuoso-large-vs-qwen3-8b

In [ ]:
run_pairwise_eval("Lakshan2003/pairwise-virtuoso-large-vs-qwen3-8b")


=== Running PAIREVAL on Lakshan2003/pairwise-virtuoso-large-vs-qwen3-8b ===
Resuming from existing processed file
Model A: virtuoso_large
Model B: qwen3_8b
Pending rows: 0


Evaluating rows: 0row [00:00, ?row/s]

Finished Lakshan2003/pairwise-virtuoso-large-vs-qwen3-8b


### virtuoso-large-vs-qwen3-1.7b

In [ ]:
run_pairwise_eval("Lakshan2003/pairwise-virtuoso-large-vs-qwen3-1.7b")


=== Running PAIREVAL on Lakshan2003/pairwise-virtuoso-large-vs-qwen3-1.7b ===
Resuming from existing processed file
Model A: virtuoso_large
Model B: qwen3_1_7b
Pending rows: 0


Evaluating rows: 0row [00:00, ?row/s]


Finished Lakshan2003/pairwise-virtuoso-large-vs-qwen3-1.7b


### virtuoso-large-vs-llama3.2-1b

In [ ]:
run_pairwise_eval("Lakshan2003/pairwise-virtuoso-large-vs-llama3.2-1b")


=== Running PAIREVAL on Lakshan2003/pairwise-virtuoso-large-vs-llama3.2-1b ===
Resuming from existing processed file
Model A: virtuoso_large
Model B: llama3_2_1b
Pending rows: 0


Evaluating rows: 0row [00:00, ?row/s]

Finished Lakshan2003/pairwise-virtuoso-large-vs-llama3.2-1b


### virtuoso-large-vs-llama3.1-8b

In [ ]:
run_pairwise_eval("Lakshan2003/pairwise-virtuoso-large-vs-llama3.1-8b")


=== Running PAIREVAL on Lakshan2003/pairwise-virtuoso-large-vs-llama3.1-8b ===
Resuming from existing processed file
Model A: virtuoso_large
Model B: llama3_1_8b
Pending rows: 0






Evaluating rows: 0row [00:00, ?row/s]

Finished Lakshan2003/pairwise-virtuoso-large-vs-llama3.1-8b


### virtuoso-large-vs-gemma-3-4b

In [ ]:
run_pairwise_eval("Lakshan2003/pairwise-virtuoso-large-vs-gemma-3-4b")


=== Running PAIREVAL on Lakshan2003/pairwise-virtuoso-large-vs-gemma-3-4b ===


README.md:   0%|          | 0.00/775 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/1.32M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Model A: virtuoso_large
Model B: gemma3_4b
Pending rows: 1000


Evaluating rows: 100%|██████████| 1000/1000 [56:17<00:00,  3.38s/row]

Finished Lakshan2003/pairwise-virtuoso-large-vs-gemma-3-4b


### virtuoso-large-vs-smollm3-3b

In [ ]:
run_pairwise_eval("Lakshan2003/pairwise-virtuoso-large-vs-smollm3-3b")


=== Running PAIREVAL on Lakshan2003/pairwise-virtuoso-large-vs-smollm3-3b ===


README.md:   0%|          | 0.00/776 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/1.29M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Model A: virtuoso_large
Model B: smollm3_3b
Pending rows: 1000


Evaluating rows: 100%|██████████| 1000/1000 [56:41<00:00,  3.40s/row]

Finished Lakshan2003/pairwise-virtuoso-large-vs-smollm3-3b


### gemini-2.5-flash-vs-gemma-3-4b

In [ ]:
run_pairwise_eval("Lakshan2003/pairwise-gemini-2.5-flash-vs-gemma-3-4b")


=== Running PAIREVAL on Lakshan2003/pairwise-gemini-2.5-flash-vs-gemma-3-4b ===


README.md:   0%|          | 0.00/777 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/1.29M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Model A: gemini_2_5_flash
Model B: gemma3_4b
Pending rows: 1000


Evaluating rows: 100%|██████████| 1000/1000 [57:27<00:00,  3.45s/row]


Finished Lakshan2003/pairwise-gemini-2.5-flash-vs-gemma-3-4b


### pairwise-gemini-2.5-flash-vs-smollm3-3b

In [ ]:
run_pairwise_eval("Lakshan2003/pairwise-gemini-2.5-flash-vs-smollm3-3b")


=== Running PAIREVAL on Lakshan2003/pairwise-gemini-2.5-flash-vs-smollm3-3b ===


README.md:   0%|          | 0.00/778 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/1.26M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Model A: gemini_2_5_flash
Model B: smollm3_3b
Pending rows: 1000


Evaluating rows: 100%|██████████| 1000/1000 [48:58<00:00,  2.94s/row]

Finished Lakshan2003/pairwise-gemini-2.5-flash-vs-smollm3-3b


### gpt4.1-vs-gemma-3-4b

In [ ]:
run_pairwise_eval("Lakshan2003/pairwise-gpt4.1-vs-gemma-3-4b")


=== Running PAIREVAL on Lakshan2003/pairwise-gpt4.1-vs-gemma-3-4b ===


README.md:   0%|          | 0.00/768 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/1.32M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Model A: gpt_4_1
Model B: gemma3_4b
Pending rows: 1000


Evaluating rows: 100%|██████████| 1000/1000 [47:15<00:00,  2.84s/row]

Finished Lakshan2003/pairwise-gpt4.1-vs-gemma-3-4b


### gpt4.1-vs-smollm3-3b

In [ ]:
run_pairwise_eval("Lakshan2003/pairwise-gpt4.1-vs-smollm3-3b")


=== Running PAIREVAL on Lakshan2003/pairwise-gpt4.1-vs-smollm3-3b ===


README.md:   0%|          | 0.00/769 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/1.29M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Model A: gpt_4_1
Model B: smollm3_3b
Pending rows: 1000


Evaluating rows: 100%|██████████| 1000/1000 [46:20<00:00,  2.78s/row]

Finished Lakshan2003/pairwise-gpt4.1-vs-smollm3-3b


In [ ]:
import os
import pandas as pd
from datasets import Dataset
from huggingface_hub import login

# CONFIG
LOCAL_DIR = "/content/drive/MyDrive/fyp-2025/Datasets/PairwiseEvalProcessed"

# UPLOADS = {
#     "pairwise-virtuoso-large-vs-qwen3-4b_processed.csv":
#         "Lakshan2003/pairwise-virtuoso-large-vs-qwen3-4b",

#     "pairwise-virtuoso-large-vs-phi-4-mini_processed.csv":
#         "Lakshan2003/pairwise-virtuoso-large-vs-phi-4-mini",

#     "pairwise-virtuoso-large-vs-llama-3.2-instruct_processed.csv":
#         "Lakshan2003/pairwise-virtuoso-large-vs-llama-3.2-instruct",

#     "pairwise-gpt4.1-vs-qwen3-4b_processed.csv":
#         "Lakshan2003/pairwise-gpt4.1-vs-qwen3-4b",

#     "pairwise-gpt4.1-vs-phi-4-mini_processed.csv":
#         "Lakshan2003/pairwise-gpt4.1-vs-phi-4-mini",

#     "pairwise-gpt4.1-vs-llama-3.2-instruct_processed.csv":
#         "Lakshan2003/pairwise-gpt4.1-vs-llama-3.2-instruct",

#     "pairwise-gemini-2.5-flash-vs-qwen3-4b_processed.csv":
#         "Lakshan2003/pairwise-gemini-2.5-flash-vs-qwen3-4b",

#     "pairwise-gemini-2.5-flash-vs-phi-4-mini_processed.csv":
#         "Lakshan2003/pairwise-gemini-2.5-flash-vs-phi-4-mini",

#     "pairwise-gemini-2.5-flash-vs-llama-3.2-instruct_processed.csv":
#         "Lakshan2003/pairwise-gemini-2.5-flash-vs-llama-3.2-instruct",
# }


UPLOADS = {
    # ========= Virtuoso-Large vs =========
    "pairwise-virtuoso-large-vs-qwen3-8b_processed.csv":
        "Lakshan2003/pairwise-virtuoso-large-vs-qwen3-8b",

    "pairwise-virtuoso-large-vs-qwen3-4b_processed.csv":
        "Lakshan2003/pairwise-virtuoso-large-vs-qwen3-4b",

    "pairwise-virtuoso-large-vs-qwen3-1.7b_processed.csv":
        "Lakshan2003/pairwise-virtuoso-large-vs-qwen3-1.7b",

    "pairwise-virtuoso-large-vs-phi-4-mini_processed.csv":
        "Lakshan2003/pairwise-virtuoso-large-vs-phi-4-mini",

    "pairwise-virtuoso-large-vs-llama3.2-1b_processed.csv":
        "Lakshan2003/pairwise-virtuoso-large-vs-llama3.2-1b",

    "pairwise-virtuoso-large-vs-llama3.1-8b_processed.csv":
        "Lakshan2003/pairwise-virtuoso-large-vs-llama3.1-8b",

    "pairwise-virtuoso-large-vs-llama-3.2-instruct_processed.csv":
        "Lakshan2003/pairwise-virtuoso-large-vs-llama-3.2-instruct",

    # 🔹 MISSING — added
    "pairwise-virtuoso-large-vs-gemma-3-4b_processed.csv":
        "Lakshan2003/pairwise-virtuoso-large-vs-gemma-3-4b",

    "pairwise-virtuoso-large-vs-smollm3-3b_processed.csv":
        "Lakshan2003/pairwise-virtuoso-large-vs-smollm3-3b",


    # ========= GPT-4.1 vs =========
    "pairwise-gpt4.1-vs-qwen3-8b_processed.csv":
        "Lakshan2003/pairwise-gpt4.1-vs-qwen3-8b",

    "pairwise-gpt4.1-vs-qwen3-4b_processed.csv":
        "Lakshan2003/pairwise-gpt4.1-vs-qwen3-4b",

    "pairwise-gpt4.1-vs-qwen3-1.7b_processed.csv":
        "Lakshan2003/pairwise-gpt4.1-vs-qwen3-1.7b",

    "pairwise-gpt4.1-vs-phi-4-mini_processed.csv":
        "Lakshan2003/pairwise-gpt4.1-vs-phi-4-mini",

    "pairwise-gpt4.1-vs-llama3.2-1b_processed.csv":
        "Lakshan2003/pairwise-gpt4.1-vs-llama3.2-1b",

    "pairwise-gpt4.1-vs-llama3.1-8b_processed.csv":
        "Lakshan2003/pairwise-gpt4.1-vs-llama3.1-8b",

    "pairwise-gpt4.1-vs-llama-3.2-instruct_processed.csv":
        "Lakshan2003/pairwise-gpt4.1-vs-llama-3.2-instruct",

    # 🔹 MISSING — added
    "pairwise-gpt4.1-vs-gemma-3-4b_processed.csv":
        "Lakshan2003/pairwise-gpt4.1-vs-gemma-3-4b",

    "pairwise-gpt4.1-vs-smollm3-3b_processed.csv":
        "Lakshan2003/pairwise-gpt4.1-vs-smollm3-3b",


    # ========= Gemini-2.5-Flash vs =========
    "pairwise-gemini-2.5-flash-vs-qwen3-8b_processed.csv":
        "Lakshan2003/pairwise-gemini-2.5-flash-vs-qwen3-8b",

    "pairwise-gemini-2.5-flash-vs-qwen3-4b_processed.csv":
        "Lakshan2003/pairwise-gemini-2.5-flash-vs-qwen3-4b",

    "pairwise-gemini-2.5-flash-vs-qwen3-1.7b_processed.csv":
        "Lakshan2003/pairwise-gemini-2.5-flash-vs-qwen3-1.7b",

    "pairwise-gemini-2.5-flash-vs-phi-4-mini_processed.csv":
        "Lakshan2003/pairwise-gemini-2.5-flash-vs-phi-4-mini",

    "pairwise-gemini-2.5-flash-vs-llama3.2-1b_processed.csv":
        "Lakshan2003/pairwise-gemini-2.5-flash-vs-llama3.2-1b",

    "pairwise-gemini-2.5-flash-vs-llama3.1-8b_processed.csv":
        "Lakshan2003/pairwise-gemini-2.5-flash-vs-llama3.1-8b",

    "pairwise-gemini-2.5-flash-vs-llama-3.2-instruct_processed.csv":
        "Lakshan2003/pairwise-gemini-2.5-flash-vs-llama-3.2-instruct",

    # 🔹 MISSING — added
    "pairwise-gemini-2.5-flash-vs-gemma-3-4b_processed.csv":
        "Lakshan2003/pairwise-gemini-2.5-flash-vs-gemma-3-4b",

    "pairwise-gemini-2.5-flash-vs-smollm3-3b_processed.csv":
        "Lakshan2003/pairwise-gemini-2.5-flash-vs-smollm3-3b",
}

# Upload loop (THIS replaces parquet correctly)
for local_file, repo_id in UPLOADS.items():
    local_path = os.path.join(LOCAL_DIR, local_file)

    if not os.path.exists(local_path):
        raise FileNotFoundError(local_path)

    print(f"\nProcessing {local_file} → {repo_id}")

    # Load processed CSV
    df = pd.read_csv(local_path)

    # Sanity check (important)
    print(df[["judge_pass_1", "judge_pass_2", "final_result"]].head())

    # Convert to HF Dataset
    dataset = Dataset.from_pandas(df, preserve_index=False)

    # Push to hub (this OVERWRITES parquet)
    dataset.push_to_hub(
        repo_id,
        split="train",
        commit_message="Replace parquet with updated PAIREVAL judge results"
    )

print("\nAll parquet datasets replaced successfully.")


Processing pairwise-virtuoso-large-vs-qwen3-8b_processed.csv → Lakshan2003/pairwise-virtuoso-large-vs-qwen3-8b
  judge_pass_1 judge_pass_2    final_result
0            B            B        qwen3_8b
1            A            A  virtuoso_large
2            A            A  virtuoso_large
3            A            A  virtuoso_large
4            B            B        qwen3_8b


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

...

New Data Upload               : |          |  0.00B /  0.00B            

                              : 100%|##########| 1.26MB / 1.26MB            

                              : 100%|##########| 1.26MB / 1.26MB            

No files have been modified since last commit. Skipping to prevent empty commit.



All parquet datasets replaced successfully.


## Summarized Pairwise Results

In [ ]:
import pandas as pd
from datasets import load_dataset, Dataset

# CONFIG
PAIRWISE_REPOS = [
    # ========= Virtuoso-Large vs =========
    "Lakshan2003/pairwise-virtuoso-large-vs-llama3.2-1b",
    "Lakshan2003/pairwise-virtuoso-large-vs-llama3.1-8b",
    "Lakshan2003/pairwise-virtuoso-large-vs-qwen3-1.7b",
    "Lakshan2003/pairwise-virtuoso-large-vs-qwen3-4b",
    "Lakshan2003/pairwise-virtuoso-large-vs-qwen3-8b",
    "Lakshan2003/pairwise-virtuoso-large-vs-phi-4-mini",
    "Lakshan2003/pairwise-virtuoso-large-vs-gemma-3-4b",
    "Lakshan2003/pairwise-virtuoso-large-vs-smollm3-3b",
    "Lakshan2003/pairwise-virtuoso-large-vs-llama-3.2-instruct",

    # ========= Gemini-2.5-Flash vs =========
    "Lakshan2003/pairwise-gemini-2.5-flash-vs-llama3.2-1b",
    "Lakshan2003/pairwise-gemini-2.5-flash-vs-llama3.1-8b",
    "Lakshan2003/pairwise-gemini-2.5-flash-vs-qwen3-1.7b",
    "Lakshan2003/pairwise-gemini-2.5-flash-vs-qwen3-4b",
    "Lakshan2003/pairwise-gemini-2.5-flash-vs-qwen3-8b",
    "Lakshan2003/pairwise-gemini-2.5-flash-vs-phi-4-mini",
    "Lakshan2003/pairwise-gemini-2.5-flash-vs-gemma-3-4b",
    "Lakshan2003/pairwise-gemini-2.5-flash-vs-smollm3-3b",
    "Lakshan2003/pairwise-gemini-2.5-flash-vs-llama-3.2-instruct",

    # ========= GPT-4.1 vs =========
    "Lakshan2003/pairwise-gpt4.1-vs-llama3.2-1b",
    "Lakshan2003/pairwise-gpt4.1-vs-llama3.1-8b",
    "Lakshan2003/pairwise-gpt4.1-vs-qwen3-1.7b",
    "Lakshan2003/pairwise-gpt4.1-vs-qwen3-4b",
    "Lakshan2003/pairwise-gpt4.1-vs-qwen3-8b",
    "Lakshan2003/pairwise-gpt4.1-vs-phi-4-mini",
    "Lakshan2003/pairwise-gpt4.1-vs-gemma-3-4b",
    "Lakshan2003/pairwise-gpt4.1-vs-smollm3-3b",
    "Lakshan2003/pairwise-gpt4.1-vs-llama-3.2-instruct",
]

SUMMARY_REPO = "Lakshan2003/pairwise-eval-results"
JUDGE_NAME = "claude-haiku-4-5"

# These are IDENTIFIERS, not full names
LLM_KEYS = ["virtuoso", "gpt4", "gemini"]


def get_models_from_columns(df: pd.DataFrame):
    """Extract the two model names exactly as used in final_result."""
    cols = [c for c in df.columns if c.startswith("generated_answer_")]
    if len(cols) != 2:
        raise ValueError(f"Expected 2 generated_answer_* columns, got {len(cols)}")

    m1 = cols[0].replace("generated_answer_", "")
    m2 = cols[1].replace("generated_answer_", "")
    return m1, m2


def classify_llm_slm(model_a: str, model_b: str):
    """Determine which model is LLM vs SLM based on known LLM keys."""
    a_is_llm = any(k in model_a for k in LLM_KEYS)
    b_is_llm = any(k in model_b for k in LLM_KEYS)

    if a_is_llm and not b_is_llm:
        return model_a, model_b
    if b_is_llm and not a_is_llm:
        return model_b, model_a

    # Fallback (should not normally happen)
    print(f"WARNING: Could not confidently classify LLM/SLM between {model_a} and {model_b}")
    return model_a, model_b


rows = []

# Aggregate results
for repo in PAIRWISE_REPOS:
    print(f"Loading {repo}")
    df = load_dataset(repo, split="train").to_pandas()

    model_a, model_b = get_models_from_columns(df)
    llm, slm = classify_llm_slm(model_a, model_b)

    llm_wins = 0
    slm_wins = 0
    ties = 0
    valid = 0

    for r in df["final_result"]:
        if r == llm:
            llm_wins += 1
            valid += 1
        elif r == slm:
            slm_wins += 1
            valid += 1
        elif r == "Tie":
            ties += 1
            valid += 1
        else:
            # ignore invalid / empty / Error rows
            continue

    rows.append({
        "llm": llm,
        "slm": slm,
        "llm_wins": llm_wins,
        "slm_wins": slm_wins,
        "ties": ties,
        "llm_win_pct": llm_wins / valid * 100 if valid else 0.0,
        "slm_win_pct": slm_wins / valid * 100 if valid else 0.0,
        "tie_pct": ties / valid * 100 if valid else 0.0,
        "tested_records": valid,
        "judge": JUDGE_NAME
    })

summary_df = (
    pd.DataFrame(rows)
    .sort_values(["llm", "slm"])
    .reset_index(drop=True)
)

for col in ["llm_win_pct", "slm_win_pct", "tie_pct"]:
    summary_df[col] = summary_df[col].round(2)

Loading Lakshan2003/pairwise-virtuoso-large-vs-llama3.2-1b
Loading Lakshan2003/pairwise-virtuoso-large-vs-llama3.1-8b
Loading Lakshan2003/pairwise-virtuoso-large-vs-qwen3-1.7b
Loading Lakshan2003/pairwise-virtuoso-large-vs-qwen3-4b
Loading Lakshan2003/pairwise-virtuoso-large-vs-qwen3-8b
Loading Lakshan2003/pairwise-virtuoso-large-vs-phi-4-mini
Loading Lakshan2003/pairwise-virtuoso-large-vs-gemma-3-4b
Loading Lakshan2003/pairwise-virtuoso-large-vs-smollm3-3b
Loading Lakshan2003/pairwise-virtuoso-large-vs-llama-3.2-instruct
Loading Lakshan2003/pairwise-gemini-2.5-flash-vs-llama3.2-1b
Loading Lakshan2003/pairwise-gemini-2.5-flash-vs-llama3.1-8b
Loading Lakshan2003/pairwise-gemini-2.5-flash-vs-qwen3-1.7b
Loading Lakshan2003/pairwise-gemini-2.5-flash-vs-qwen3-4b
Loading Lakshan2003/pairwise-gemini-2.5-flash-vs-qwen3-8b
Loading Lakshan2003/pairwise-gemini-2.5-flash-vs-phi-4-mini
Loading Lakshan2003/pairwise-gemini-2.5-flash-vs-gemma-3-4b
Loading Lakshan2003/pairwise-gemini-2.5-flash-vs-smoll

In [ ]:
DISPLAY_NAME_MAP = {
    # LLMs (no parameter count)
    "virtuoso_large": "Virtuoso-Large",
    "gpt4_1": "GPT-4.1",
    "gemini_2_5_flash": "Gemini-2.5-Flash",
    "gemma3_4b": "Gemma-3-4B",

    # SLMs (explicit parameter counts)
    "qwen3_4b": "Qwen-3-4B-Instruct (4B)",
    "phi_4_mini": "Phi-4-Mini (4B)",
    "llama3_2": "LLaMA-3.2-3B",
    "llama3_2_instruct": "LLaMA-3.2-3B-Instruct",
    "smollm3_3b": "SmolLM-3B-Instruct",
}


def to_display(name):
    return DISPLAY_NAME_MAP.get(name, name)

summary_df["llm"] = summary_df["llm"].apply(to_display)
summary_df["slm"] = summary_df["slm"].apply(to_display)

In [ ]:
summary_df.loc[0, "slm_win_pct"]

np.float64(6.3)

In [ ]:
print("\nFinal LLM vs SLM results table (with percentages):")
summary_df


Final LLM vs SLM results table (with percentages):


,llm,slm,llm_wins,slm_wins,ties,llm_win_pct,slm_win_pct,tie_pct,tested_records,judge
0,Gemini-2.5-Flash,Gemma-3-4B,885,63,52,88.50,6.30,5.20,1000,claude-haiku-4-5
1,Gemini-2.5-Flash,llama3_1_8b,286,529,185,28.60,52.90,18.50,1000,claude-haiku-4-5
2,Gemini-2.5-Flash,LLaMA-3.2-3B,374,497,129,37.40,49.70,12.90,1000,claude-haiku-4-5
3,Gemini-2.5-Flash,llama3_2_1b,431,386,183,43.10,38.60,18.30,1000,claude-haiku-4-5
4,Gemini-2.5-Flash,Phi-4-Mini (4B),405,439,156,40.50,43.90,15.60,1000,claude-haiku-4-5
5,Gemini-2.5-Flash,qwen3_1_7b,498,335,167,49.80,33.50,16.70,1000,claude-haiku-4-5
6,Gemini-2.5-Flash,Qwen-3-4B-Instruct (4B),412,450,138,41.20,45.00,13.80,1000,claude-haiku-4-5
7,Gemini-2.5-Flash,qwen3_8b,266,558,176,26.60,55.80,17.60,1000,claude-haiku-4-5
8,Gemini-2.5-Flash,SmolLM-3B-Instruct,858,78,64,85.80,7.80,6.40,1000,claude-haiku-4-5
9,gpt_4_1,Gemma-3-4B,973,15,12,97.30,1.50,1.20,1000,claude-haiku-4-5


In [ ]:
summary_df.drop(columns=["llm_wins", "slm_wins", "ties"])

,llm,slm,llm_win_pct,slm_win_pct,tie_pct,tested_records,judge
0,Gemini-2.5-Flash,Gemma-3-4B,88.5,6.3,5.2,1000,claude-haiku-4-5
1,Gemini-2.5-Flash,llama3_1_8b,28.6,52.9,18.5,1000,claude-haiku-4-5
2,Gemini-2.5-Flash,LLaMA-3.2-3B,37.4,49.7,12.9,1000,claude-haiku-4-5
3,Gemini-2.5-Flash,llama3_2_1b,43.1,38.6,18.3,1000,claude-haiku-4-5
4,Gemini-2.5-Flash,Phi-4-Mini (4B),40.5,43.9,15.6,1000,claude-haiku-4-5
5,Gemini-2.5-Flash,qwen3_1_7b,49.8,33.5,16.7,1000,claude-haiku-4-5
6,Gemini-2.5-Flash,Qwen-3-4B-Instruct (4B),41.2,45.0,13.8,1000,claude-haiku-4-5
7,Gemini-2.5-Flash,qwen3_8b,26.6,55.8,17.6,1000,claude-haiku-4-5
8,Gemini-2.5-Flash,SmolLM-3B-Instruct,85.8,7.8,6.4,1000,claude-haiku-4-5
9,gpt_4_1,Gemma-3-4B,97.3,1.5,1.2,1000,claude-haiku-4-5


In [ ]:
summary_ds = Dataset.from_pandas(summary_df, preserve_index=False)

summary_ds.push_to_hub(
    SUMMARY_REPO,
    split="train",
    commit_message="Corrected LLM vs SLM pairwise results with percentages (Claude Sonnet judge)"
)

print(f"\nSummary pushed to {SUMMARY_REPO}")

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              : 100%|##########| 5.32kB / 5.32kB            

README.md:   0%|          | 0.00/593 [00:00<?, ?B/s]


Summary pushed to Lakshan2003/pairwise-eval-results


### Stag-wise Pairwise Evaluation Results

In [ ]:
import pandas as pd
from datasets import load_dataset

# CONFIG (unchanged)
PAIRWISE_REPOS = [
    "Lakshan2003/pairwise-virtuoso-large-vs-llama3.2-1b",
    "Lakshan2003/pairwise-virtuoso-large-vs-llama3.1-8b",
    "Lakshan2003/pairwise-virtuoso-large-vs-qwen3-1.7b",
    "Lakshan2003/pairwise-virtuoso-large-vs-qwen3-4b",
    "Lakshan2003/pairwise-virtuoso-large-vs-qwen3-8b",
    "Lakshan2003/pairwise-virtuoso-large-vs-phi-4-mini",
    "Lakshan2003/pairwise-virtuoso-large-vs-gemma-3-4b",
    "Lakshan2003/pairwise-virtuoso-large-vs-smollm3-3b",
    "Lakshan2003/pairwise-virtuoso-large-vs-llama-3.2-instruct",

    "Lakshan2003/pairwise-gemini-2.5-flash-vs-llama3.2-1b",
    "Lakshan2003/pairwise-gemini-2.5-flash-vs-llama3.1-8b",
    "Lakshan2003/pairwise-gemini-2.5-flash-vs-qwen3-1.7b",
    "Lakshan2003/pairwise-gemini-2.5-flash-vs-qwen3-4b",
    "Lakshan2003/pairwise-gemini-2.5-flash-vs-qwen3-8b",
    "Lakshan2003/pairwise-gemini-2.5-flash-vs-phi-4-mini",
    "Lakshan2003/pairwise-gemini-2.5-flash-vs-gemma-3-4b",
    "Lakshan2003/pairwise-gemini-2.5-flash-vs-smollm3-3b",
    "Lakshan2003/pairwise-gemini-2.5-flash-vs-llama-3.2-instruct",

    "Lakshan2003/pairwise-gpt4.1-vs-llama3.2-1b",
    "Lakshan2003/pairwise-gpt4.1-vs-llama3.1-8b",
    "Lakshan2003/pairwise-gpt4.1-vs-qwen3-1.7b",
    "Lakshan2003/pairwise-gpt4.1-vs-qwen3-4b",
    "Lakshan2003/pairwise-gpt4.1-vs-qwen3-8b",
    "Lakshan2003/pairwise-gpt4.1-vs-phi-4-mini",
    "Lakshan2003/pairwise-gpt4.1-vs-gemma-3-4b",
    "Lakshan2003/pairwise-gpt4.1-vs-smollm3-3b",
    "Lakshan2003/pairwise-gpt4.1-vs-llama-3.2-instruct",
]

JUDGE_NAME = "claude-haiku-4-5"
LLM_KEYS = ["virtuoso", "gpt4", "gemini"]


def get_models_from_columns(df):
    cols = [c for c in df.columns if c.startswith("generated_answer_")]
    if len(cols) != 2:
        raise ValueError(f"Expected 2 generated_answer_* columns, got {len(cols)}")
    return cols[0].replace("generated_answer_", ""), cols[1].replace("generated_answer_", "")


def classify_llm_slm(model_a, model_b):
    a_is_llm = any(k in model_a for k in LLM_KEYS)
    b_is_llm = any(k in model_b for k in LLM_KEYS)

    if a_is_llm and not b_is_llm:
        return model_a, model_b
    if b_is_llm and not a_is_llm:
        return model_b, model_a

    print(f"WARNING: Could not confidently classify {model_a} vs {model_b}")
    return model_a, model_b


rows = []

for repo in PAIRWISE_REPOS:
    print(f"Loading {repo}")
    df = load_dataset(repo, split="train").to_pandas()

    model_a, model_b = get_models_from_columns(df)
    llm, slm = classify_llm_slm(model_a, model_b)

    for stage, sdf in df.groupby("conversation_stage"):
        llm_wins = 0
        slm_wins = 0
        ties = 0
        valid = 0

        for r in sdf["final_result"]:
            if r == llm:
                llm_wins += 1
                valid += 1
            elif r == slm:
                slm_wins += 1
                valid += 1
            elif r == "Tie":
                ties += 1
                valid += 1

        if valid == 0:
            continue

        rows.append({
            "conversation_stage": stage,
            "llm": llm,
            "slm": slm,
            "llm_wins": llm_wins,
            "slm_wins": slm_wins,
            "ties": ties,
            "llm_win_pct": round(llm_wins / valid * 100, 2),
            "slm_win_pct": round(slm_wins / valid * 100, 2),
            "tie_pct": round(ties / valid * 100, 2),
            "tested_records": valid,
            "judge": JUDGE_NAME
        })


stagewise_summary_df = (
    pd.DataFrame(rows)
    .sort_values(["conversation_stage", "llm", "slm"])
    .reset_index(drop=True)
)

Loading Lakshan2003/pairwise-virtuoso-large-vs-llama3.2-1b
Loading Lakshan2003/pairwise-virtuoso-large-vs-llama3.1-8b
Loading Lakshan2003/pairwise-virtuoso-large-vs-qwen3-1.7b
Loading Lakshan2003/pairwise-virtuoso-large-vs-qwen3-4b
Loading Lakshan2003/pairwise-virtuoso-large-vs-qwen3-8b
Loading Lakshan2003/pairwise-virtuoso-large-vs-phi-4-mini
Loading Lakshan2003/pairwise-virtuoso-large-vs-gemma-3-4b
Loading Lakshan2003/pairwise-virtuoso-large-vs-smollm3-3b
Loading Lakshan2003/pairwise-virtuoso-large-vs-llama-3.2-instruct
Loading Lakshan2003/pairwise-gemini-2.5-flash-vs-llama3.2-1b
Loading Lakshan2003/pairwise-gemini-2.5-flash-vs-llama3.1-8b
Loading Lakshan2003/pairwise-gemini-2.5-flash-vs-qwen3-1.7b
Loading Lakshan2003/pairwise-gemini-2.5-flash-vs-qwen3-4b
Loading Lakshan2003/pairwise-gemini-2.5-flash-vs-qwen3-8b
Loading Lakshan2003/pairwise-gemini-2.5-flash-vs-phi-4-mini
Loading Lakshan2003/pairwise-gemini-2.5-flash-vs-gemma-3-4b
Loading Lakshan2003/pairwise-gemini-2.5-flash-vs-smoll

In [ ]:
pivot_mid = (
    stagewise_summary_df
    .query("conversation_stage == 'MIDDLE'")
    .pivot(index="slm", columns="llm", values="slm_win_pct")
)

In [ ]:
pd.set_option('display.max_rows', None)        # Show all rows
pd.set_option('display.max_columns', None)     # Show all columns
pd.set_option('display.width', None)           # Don't wrap to next line
pd.set_option('display.max_colwidth', None)    # Show full column content

In [ ]:
stagewise_summary_df.drop(columns=["tested_records", "llm_wins", "slm_wins", "ties"], inplace=True)

In [ ]:
stagewise_summary_df.head(100)

,conversation_stage,llm,slm,llm_win_pct,slm_win_pct,tie_pct,judge
0,BEGINNING,gemini_2_5_flash,gemma3_4b,93.94,4.04,2.02,claude-haiku-4-5
1,BEGINNING,gemini_2_5_flash,llama3_1_8b,31.31,40.40,28.28,claude-haiku-4-5
2,BEGINNING,gemini_2_5_flash,llama3_2,42.42,44.44,13.13,claude-haiku-4-5
3,BEGINNING,gemini_2_5_flash,llama3_2_1b,49.49,35.35,15.15,claude-haiku-4-5
4,BEGINNING,gemini_2_5_flash,phi_4_mini,45.45,37.37,17.17,claude-haiku-4-5
5,BEGINNING,gemini_2_5_flash,qwen3_1_7b,56.57,28.28,15.15,claude-haiku-4-5
6,BEGINNING,gemini_2_5_flash,qwen3_4b,49.49,36.36,14.14,claude-haiku-4-5
7,BEGINNING,gemini_2_5_flash,qwen3_8b,44.44,38.38,17.17,claude-haiku-4-5
8,BEGINNING,gemini_2_5_flash,smollm3_3b,92.93,5.05,2.02,claude-haiku-4-5
9,BEGINNING,gpt_4_1,gemma3_4b,97.98,1.01,1.01,claude-haiku-4-5
